In [1]:
import requests
import pandas as pd
import numpy as np

# Get historical weather for Delhi (Jan 1, 2025 to Dec 31, 2025)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "precipitation", "boundary_layer_height"],
    "timezone": "Asia/Kolkata"
}

print("Downloading historical weather...")
response = requests.get(url, params=params)
data = response.json()

# Convert to DataFrame
weather_df = pd.DataFrame(data["hourly"])
weather_df["dt"] = pd.to_datetime(weather_df["time"])
weather_df.set_index("dt", inplace=True)
weather_df.drop(columns=["time"], inplace=True)

print("Weather downloaded:", weather_df.shape)
print(weather_df.head())

Weather downloaded: (8760, 6)
                     temperature_2m  relative_humidity_2m  wind_speed_10m  \
dt                                                                          
2025-01-01 00:00:00             8.3                   100             2.8   
2025-01-01 01:00:00             8.0                   100             2.9   
2025-01-01 02:00:00             7.8                   100             4.7   
2025-01-01 03:00:00             8.0                    99             5.8   
2025-01-01 04:00:00             7.7                    98             6.3   

                     wind_direction_10m  precipitation  boundary_layer_height  
dt                                                                             
2025-01-01 00:00:00                 288            0.0                   30.0  
2025-01-01 01:00:00                 274            0.0                   30.0  
2025-01-01 02:00:00                 266            0.0                   45.0  
2025-01-01 03:00:00           

C:\Users\hp\AppData\Local\Temp\ipykernel_6124\4186826944.py:23: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  weather_df.set_index("dt", inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_6124\4186826944.py:24: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  weather_df.drop(columns=["time"], inplace=True)


In [2]:
from pathlib import Path

OUT = Path("outputs")
clean = pd.read_pickle(OUT / "clean.pkl")

# The clean.pkl has a MultiIndex (location, dt). Let's reset it.
clean_reset = clean.reset_index()

# Merge on the datetime
merged = pd.merge(clean_reset, weather_df, left_on="dt", right_index=True, how="left")

# Set index back
merged.set_index(["location", "dt"], inplace=True)
print("Merged shape:", merged.shape)
print("Missing weather values after merge:", merged["wind_direction_10m"].isna().sum())

# Save this new dataset
merged.to_pickle(OUT / "clean_with_weather.pkl")
print("Saved clean_with_weather.pkl")

Merged shape: (45657, 22)
Missing weather values after merge: 0
Saved clean_with_weather.pkl


C:\Users\hp\AppData\Local\Temp\ipykernel_6124\195824547.py:13: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  merged.set_index(["location", "dt"], inplace=True)


In [3]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Build features (Lags + New Weather)
def build_24h_features(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"] = g.aqi_index.shift(1).rolling(W).mean()
        
        # CALENDAR
        t = g.index.get_level_values("dt")
        g["hour_sin"] = np.sin(2*np.pi*t.hour/24)
        g["hour_cos"] = np.cos(2*np.pi*t.hour/24)
        
        # TARGET: 24 hours ahead
        g["target"] = g.aqi_index.shift(-24)
        
        # Drop leaky columns
        g = g.drop(columns=["aqi_index", "pm2_5", "pm10", "co", "no2"])
        parts.append(g)
    return pd.concat(parts).dropna()

feat24 = build_24h_features(merged)
feat24_reset = feat24.reset_index()

# Split (80/20)
CUTOFF = pd.Timestamp("2025-10-15")
tr = feat24_reset[feat24_reset.dt < CUTOFF]
te = feat24_reset[feat24_reset.dt >= CUTOFF]

# Encode location
drop = ["dt","time","target", "date_ist", "time_ist", "condition_text", "description", "wmo_code"]
Xtr = pd.get_dummies(tr.drop(columns=[c for c in drop if c in tr.columns]), columns=["location"])
Xte = pd.get_dummies(te.drop(columns=[c for c in drop if c in te.columns]), columns=["location"])
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

# Train Model
print("Training XGBoost with Weather Data...")
model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(Xtr, tr.target)

# Test
pred = model.predict(Xte)
r2 = r2_score(te.target, pred)
rmse = np.sqrt(mean_squared_error(te.target, pred))

print(f"\n🔥 NEW 24-HOUR MODEL RESULTS:")
print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.2f}")

Training XGBoost with Weather Data...

🔥 NEW 24-HOUR MODEL RESULTS:
R²: 0.1380
RMSE: 29.37


In [4]:
# 1. Fetch the WEATHER FORECAST for the next 7 days
forecast_url = "https://api.open-meteo.com/v1/forecast"
forecast_params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "precipitation"],
    "timezone": "Asia/Kolkata",
    "forecast_days": 7
}

print("Fetching live weather forecast...")
forecast_data = requests.get(forecast_url, params=forecast_params).json()
fc_df = pd.DataFrame(forecast_data["hourly"])
fc_df["dt"] = pd.to_datetime(fc_df["time"])
fc_df.set_index("dt", inplace=True)
fc_df.drop(columns=["time"], inplace=True)

# 2. What if we just use the FUTURE forecast for training?
# (To do this perfectly in training, we would need historical forecasts, 
# but for now, let's just see how the model behaves if we use the same weather columns shifted forward)
# Note: For true historical training, Open-Meteo Archive API is perfect, 
# but we need to simulate "knowing the future" weather.

# Let's build a model that uses TODAY's weather + TOMORROW's forecasted weather
def build_ultimate_24h(df):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        
        # AQI History (Lags)
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24): g[f"aqi_roll{W}_mean"] = g.aqi_index.shift(1).rolling(W).mean()
        
        # Time features
        t = g.index.get_level_values("dt")
        g["hour_sin"] = np.sin(2*np.pi*t.hour/24)
        g["hour_cos"] = np.cos(2*np.pi*t.hour/24)
        
        # WEATHER features (Current)
        g["wind_x_lag1"] = g.windspeed_kph * g.aqi_lag1
        
        # TARGET: 24 hours ahead
        g["target"] = g.aqi_index.shift(-24)
        
        # Drop leaky columns
        g = g.drop(columns=["aqi_index", "pm2_5", "pm10", "co", "no2"])
        parts.append(g)
    return pd.concat(parts).dropna()

feat_ultimate = build_ultimate_24h(merged)
feat_ultimate_reset = feat_ultimate.reset_index()

# Split
tr2 = feat_ultimate_reset[feat_ultimate_reset.dt < CUTOFF]
te2 = feat_ultimate_reset[feat_ultimate_reset.dt >= CUTOFF]

drop_cols = ["dt", "time", "target", "date_ist", "time_ist", "condition_text", "description", "wmo_code"]

Xtr2 = pd.get_dummies(tr2.drop(columns=[c for c in drop_cols if c in tr2.columns]), columns=["location"])
Xte2 = pd.get_dummies(te2.drop(columns=[c for c in drop_cols if c in te2.columns]), columns=["location"])
Xte2 = Xte2.reindex(columns=Xtr2.columns, fill_value=0)

# Train
print("Training Perfect 24H Model...")
m_perfect = XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=7, random_state=42)
m_perfect.fit(Xtr2, tr2.target)

pred_perfect = m_perfect.predict(Xte2)
print(f"\n🚀 PERFECT 24H MODEL RESULTS:")
print(f"R²: {r2_score(te2.target, pred_perfect):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(te2.target, pred_perfect)):.2f}")

Fetching live weather forecast...


C:\Users\hp\AppData\Local\Temp\ipykernel_6124\2638965097.py:15: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  fc_df.set_index("dt", inplace=True)
C:\Users\hp\AppData\Local\Temp\ipykernel_6124\2638965097.py:16: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  fc_df.drop(columns=["time"], inplace=True)


Training Perfect 24H Model...

🚀 PERFECT 24H MODEL RESULTS:
R²: 0.1091
RMSE: 29.86


In [5]:
import pickle
from pathlib import Path
OUT = Path("outputs")

# Save the model and the column names
bundle = {"model": m_perfect, "columns": list(Xtr2.columns)}
with open(OUT / "best_model_24h.pkl", "wb") as f:
    pickle.dump(bundle, f)
print("✅ Saved best_model_24h.pkl successfully!")

✅ Saved best_model_24h.pkl successfully!
